# TEP Data Preprocessing — First Draft

**Maryam Ali · 202509001 · MSc Artificial Intelligence, Bahrain Polytechnic**

This notebook covers the data loading and preprocessing pipeline for the Tennessee Eastman
Process (TEP) anomaly detection thesis. Modelling notebooks build on this pipeline separately.

**Scope of this draft:**
1. Environment setup
2. Data loading (`.RData` → CSV, with cache validation)
3. Preprocessing (binary framing, train/test split, scaling)

> Status: **Working draft** — baseline data pipeline only. Model training notebooks to follow.

## 1 · Environment Setup

In [1]:
# Uncomment on first run:
# %pip install pyreadr scikit-learn pandas numpy -q

import os, gc, time
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

SEED = 42
np.random.seed(SEED)

for d in ("csv",):
    os.makedirs(d, exist_ok=True)

print("Environment ready.")

Environment ready.


## 2 · Data Loading — RData → CSV

The TEP dataset ships as `.RData` files. This section converts each to CSV **once** and
caches the result; later runs load from cache for speed.

| File | Content | Rows |
|---|---|---|
| `TEP_FaultFree_Training.RData` | Normal operation — 500 runs × 500 samples | 250,000 |
| `TEP_FaultFree_Testing.RData` | Normal operation — 500 runs × 960 samples | 480,000 |
| `TEP_Faulty_Training.RData` | Faults 1–20 — 500 runs × 500 samples each | 5,000,000 |

Each row: 41 measured variables (`xmeas_*`) + 11 manipulated variables (`xmv_*`), plus
`faultNumber`, `simulationRun`, `sample`.

> **Cache validation built in:** the loader checks required columns are present on
> *every* load — cache or fresh — and automatically rebuilds a corrupted cache rather
> than silently returning incomplete data.

In [4]:
import pyreadr

DATA_DIR = "."          # folder containing the .RData files
ROWS_PER_FAULT = 15_000  # cap per fault class to keep the file manageable

FEATURES = [f"xmeas_{i}" for i in range(1, 42)] + [f"xmv_{i}" for i in range(1, 12)]
META = ["faultNumber", "simulationRun", "sample"]
REQUIRED_COLS = set(META) - {"faultNumber"}


def _pick_dataframe(rdata_path: str, expect_fault_col: bool) -> pd.DataFrame:
    result = pyreadr.read_r(rdata_path)
    print(f"    (RData contains {len(result)} object(s): {list(result.keys())})")

    if expect_fault_col:
        for name, obj in result.items():
            if "faultNumber" in obj.columns:
                print(f"    (using object '{name}' — contains faultNumber)")
                return obj
        for name, obj in result.items():
            print(f"    Object '{name}' columns: {list(obj.columns)}")
        raise ValueError(f"No object in {rdata_path} contains a 'faultNumber' column.")

    return next(iter(result.values()))


def _sample_per_fault(df: pd.DataFrame, rows_per_fault: int) -> pd.DataFrame:
    """Stratified sampling by fault class WITHOUT using groupby().apply() —
    that pattern silently drops the grouping column in recent pandas versions.
    A plain loop + concat preserves every column reliably.
    """
    parts = []
    for fault_val, group in df.groupby("faultNumber"):
        n = min(len(group), rows_per_fault)
        parts.append(group.sample(n=n, random_state=SEED))
    return pd.concat(parts, ignore_index=True)


def load_tep(rdata_name: str, csv_name: str, rows_per_fault: int | None = None,
             expect_fault_col: bool = False, force_reload: bool = False) -> pd.DataFrame | None:
    csv_path = os.path.join("csv", csv_name)

    def _validate(df):
        missing = [c for c in REQUIRED_COLS if c not in df.columns]
        if expect_fault_col and "faultNumber" not in df.columns:
            missing.append("faultNumber")
        return missing

    if force_reload and os.path.exists(csv_path):
        print(f"  ↻ {csv_name:<28} force_reload=True — discarding cache")
        os.remove(csv_path)

    if os.path.exists(csv_path):
        df = pd.read_csv(csv_path)
        missing = _validate(df)
        if not missing:
            print(f"  ✓ {csv_name:<28} loaded from cache   {df.shape}")
            return df
        else:
            print(f"  ⚠ {csv_name:<28} cached file missing {missing} — rebuilding from .RData")
            os.remove(csv_path)

    rdata_path = os.path.join(DATA_DIR, rdata_name)
    if not os.path.exists(rdata_path):
        print(f"  ✗ {rdata_name:<28} not found at '{rdata_path}' — skipped")
        return None

    t0 = time.time()
    df = _pick_dataframe(rdata_path, expect_fault_col)

    missing = _validate(df)
    if missing:
        raise ValueError(f"{rdata_name} is missing required columns {missing} even on a fresh load.")

    df[FEATURES] = df[FEATURES].astype("float32")

    if rows_per_fault is not None and "faultNumber" in df.columns:
        df = _sample_per_fault(df, rows_per_fault)
        assert "faultNumber" in df.columns, "faultNumber was lost during sampling — this should not happen."

    df.to_csv(csv_path, index=False)
    print(f"  ✓ {rdata_name:<28} converted in {time.time()-t0:.0f}s → {df.shape}")
    gc.collect()
    return df


print("Loading TEP dataset …")
df_normal_train = load_tep("TEP_FaultFree_Training.RData", "fault_free_training.csv")
df_faulty_train = load_tep("TEP_Faulty_Training.RData",    "faulty_training.csv",
                            ROWS_PER_FAULT, expect_fault_col=True, force_reload=True)

assert df_normal_train is not None, "Fault-free training file is required."
assert df_faulty_train is not None, "Faulty training file is required."
assert "faultNumber" in df_faulty_train.columns, "faultNumber missing — see diagnostic output above."
print("\nBoth files loaded and validated ✓")

Loading TEP dataset …
  ✓ fault_free_training.csv      loaded from cache   (250000, 55)
  ↻ faulty_training.csv          force_reload=True — discarding cache
    (RData contains 1 object(s): ['faulty_training'])
    (using object 'faulty_training' — contains faultNumber)
  ✓ TEP_Faulty_Training.RData    converted in 31s → (300000, 55)

Both files loaded and validated ✓


In [5]:
# Quick integrity check
na_total = df_normal_train[FEATURES].isna().sum().sum() + df_faulty_train[FEATURES].isna().sum().sum()
print(f"Missing values across both training sets: {na_total}")
print(f"Fault classes present: {sorted(df_faulty_train['faultNumber'].unique())}")
print(f"\nNormal training shape: {df_normal_train.shape}")
print(f"Faulty training shape:  {df_faulty_train.shape}")

Missing values across both training sets: 0
Fault classes present: [np.int32(1), np.int32(2), np.int32(3), np.int32(4), np.int32(5), np.int32(6), np.int32(7), np.int32(8), np.int32(9), np.int32(10), np.int32(11), np.int32(12), np.int32(13), np.int32(14), np.int32(15), np.int32(16), np.int32(17), np.int32(18), np.int32(19), np.int32(20)]

Normal training shape: (250000, 55)
Faulty training shape:  (300000, 55)


## 3 · Preprocessing

Binary framing: **0 = normal**, **1 = any fault** — the operational question a DCS
operator actually faces.

- Stratified 70/30 train/test split
- `StandardScaler` fit on training data only (no leakage)
- A **normal-only** view of the training set is kept aside for later semi-supervised modelling

In [6]:
N_NORMAL = min(len(df_normal_train), 500_000)

normal = df_normal_train.sample(N_NORMAL, random_state=SEED)[FEATURES + META].copy()
normal["label"] = 0
faulty = df_faulty_train[FEATURES + META].copy()
faulty["label"] = 1

data = pd.concat([normal, faulty], ignore_index=True)

X, y = data[FEATURES].values.astype("float32"), data["label"].values
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=SEED)

scaler = StandardScaler().fit(X_train)
X_train_s, X_test_s = scaler.transform(X_train), scaler.transform(X_test)
X_train_normal = X_train_s[y_train == 0]        # reserved for future semi-supervised modelling

print(f"Train: {X_train_s.shape} · Test: {X_test_s.shape}")
print(f"Class balance (train): normal={np.sum(y_train==0):,}  fault={np.sum(y_train==1):,}")
print(f"Class balance (test):  normal={np.sum(y_test==0):,}  fault={np.sum(y_test==1):,}")

del data, X, y; gc.collect()

Train: (385000, 52) · Test: (165000, 52)
Class balance (train): normal=175,000  fault=210,000
Class balance (test):  normal=75,000  fault=90,000


0

In [7]:
# Save preprocessed arrays for downstream modelling notebooks
np.save("csv/X_train_scaled.npy", X_train_s)
np.save("csv/X_test_scaled.npy", X_test_s)
np.save("csv/y_train.npy", y_train)
np.save("csv/y_test.npy", y_test)
np.save("csv/X_train_normal_only.npy", X_train_normal)

print("Preprocessed arrays saved to csv/ — ready for model training notebooks.")

Preprocessed arrays saved to csv/ — ready for model training notebooks.
